# IaC 第3周:Ansible — 配置管理与自动化

> **学习目标**:理解命令式配置管理,能用 Ansible 管理服务器配置,知道 Terraform 和 Ansible 的分工边界

---

## Day 15:Terraform vs Ansible 分工

| 维度 | Terraform | Ansible |
|------|-----------|----------|
| 范式 | 声明式 (what) | 命令式 (how) |
| 管理对象 | 基础设施 (VM/网络/存储) | 服务器上的软件和配置 |
| 状态 | 有 state 文件 | 无 state (幂等性靠模块保证) |
| 连接方式 | API 调用 (云厂商 API) | SSH 连接 (无 agent) |

### 组合方式

```
Terraform 创建 3 台虚拟机 → 输出 IP 列表
    ↓
Ansible 读取 IP 列表 (inventory)
    ↓
Ansible 在 3 台机器上安装 Docker、配置用户、部署监控 agent
```

## Day 16:Inventory — 主机清单

```yaml
# inventory.yml
all:
  children:
    webservers:
      hosts:
        web-01:
          ansible_host: 10.0.0.1
          ansible_user: ubuntu
    databases:
      hosts:
        db-01:
          ansible_host: 10.0.0.10
  vars:
    ansible_python_interpreter: /usr/bin/python3
```

## Day 17:常用模块

| 模块 | 用途 | 示例 |
|------|------|------|
| command | 执行命令 (不经 shell) | `ansible all -m command -a 'uptime'` |
| shell | 执行 shell (管道/重定向) | `ansible all -m shell -a 'df -h | grep /$'` |
| copy | 复制文件到远程 | `ansible all -m copy -a 'src=a.txt dest=/tmp/'` |
| template | 渲染 Jinja2 模板 | `ansible all -m template -a 'src=nginx.conf.j2 dest=/etc/nginx/'` |
| apt/yum | 包管理 | `ansible all -m apt -a 'name=nginx state=present' --become` |
| service | 管理 systemd 服务 | `ansible all -m service -a 'name=nginx state=started enabled=yes'` |
| user | 管理用户 | `ansible all -m user -a 'name=deploy groups=docker' --become` |

## Day 18:Playbook — YAML 剧本

```yaml
# site.yml
---
- name: Configure web servers      ← Play
  hosts: webservers
  become: yes
  vars:
    nginx_port: 80

  tasks:
    - name: Install nginx
      apt:
        name: nginx
        state: present

    - name: Copy nginx config
      template:
        src: nginx.conf.j2
        dest: /etc/nginx/nginx.conf
      notify: restart nginx        ← 通知 handler

  handlers:                         ← 被 notify 触发
    - name: restart nginx
      service:
        name: nginx
        state: restarted
```

## Day 19:Jinja2 模板与 Facts

```jinja2
{# nginx.conf.j2 #}
worker_processes {{ ansible_processor_vcpus | default(1) }};
server {
    listen {{ nginx_port }};
    {% if environment == "prod" %}
    access_log /var/log/nginx/access.log;
    {% else %}
    access_log /var/log/nginx/access.log debug;
    {% endif %}
}
```

常用 Facts(自动收集):
- `{{ ansible_facts['os_family'] }}` → Debian / RedHat
- `{{ ansible_facts['processor_vcpus'] }}` → CPU 核数
- `{{ ansible_facts['memtotal_mb'] }}` → 总内存 MB

## Day 20:Roles — Ansible 的 Module

```
roles/nginx/
├── tasks/main.yml         # 核心任务
├── handlers/main.yml      # notify 触发的 handler
├── templates/             # Jinja2 模板文件
├── files/                 # 静态文件
├── vars/main.yml          # 高优先级变量
├── defaults/main.yml      # 低优先级默认变量
└── meta/main.yml          # 元信息 (依赖等)
```

```bash
ansible-galaxy init roles/nginx  # 脚手架生成 role
```

变量优先级(从低到高):role defaults → inventory vars → play vars → role vars → CLI extra vars

## Day 21:第3周综合练习

In [ ]:
print("=" * 60)
print("第3周综合练习交付清单")
print("=" * 60)

print("""
项目: ansible-server-setup/
├── ansible.cfg                 # 配置
├── inventory.yml               # 主机清单
├── site.yml                    # 主 Playbook
├── group_vars/
│   └── all.yml                 # 全局变量
├── roles/
│   ├── common/                 # 基础配置 (用户/SSH/时区)
│   ├── devtools/               # 开发工具 (Python/Docker/Vim/Tmux)
│   └── monitoring/             # 监控 agent
└── requirements.yml            # galaxy 依赖

要求:
  - 用 Jinja2 + Facts 适配不同 OS (Debian/RedHat)
  - 所有操作幂等 (重复执行安全)
  - ansible-playbook site.yml --check 通过
  - 使用 ansible-lint 检查代码质量
""")

print("=" * 60)
print("第3周核心收获:")
print("1. Terraform 管基础设施 (VM/网络),Ansible 管机器上的配置 (软件/用户)")
print("2. Inventory = 主机清单;Playbook = 任务剧本;Module = 执行单元")
print("3. Handler 被 notify 触发 (配置变了才重启服务)")
print("4. Role = Ansible 的 Module,可复用的任务包")
print("5. Jinja2 + Facts → 同一套 Playbook 适配不同 OS")
print("=" * 60)